# 7.1. Остывание стержня — можно ли доверять устойчивой схеме?

Рассматривается одномерное уравнение теплопроводности на отрезке $x \in [0, 1]$ с однородными граничными условиями Дирихле (стержень с теплоизолированной боковой поверхностью и нулевой температурой на концах):
$$
u_t = u_{xx}, \qquad x \in [0, 1], \quad t > 0,
$$
$$
u(0, t) = 0, \qquad u(1, t) = 0.
$$

В работе исследуются два класса начальных условий:
1. **Гладкое начальное распределение** (зелёный уровень):
   $$
   u(x, 0) = \sin(\pi x) + 0.3\sin(5\pi x).
   $$
   Аналитическое решение находится методом разделения переменных (ряды Фурье):
   $$
   u(x, t) = e^{-\pi^2 t}\sin(\pi x) + 0.3 e^{-25\pi^2 t}\sin(5\pi x).
   $$
   Декременты затухания гармоник $\lambda_1 = \pi^2 \approx 9.87$ и $\lambda_5 = 25\pi^2 \approx 246.74$ различаются в 25 раз, что позволяет наглядно проследить затухание мелкомасштабных пространственных деталей.

2. **Разрывное начальное распределение** (жёлтый уровень):
   $$
   u(x, 0) = \begin{cases} 1, & 0.4 \le x \le 0.6, \\ 0, & \text{иначе}. \end{cases}
   $$
   Ступенчатый профиль содержит бесконечный спектр гармоник с медленно убывающими коэффициентами Фурье ($O(1/k)$), что порождает эффект Гиббса и выявляет паразитные численные осцилляции в схемах второго порядка по времени.

### Структура исследования:
1:
   - Реализация явной схемы FTCS и неявной схемы Эйлера при параметре Куранта $r = \Delta t / \Delta x^2 = 0.4$.
   - Сопоставление с точным решением при $t \in \{0.001, 0.01, 0.1\}$ на сетках $N \in \{25, 50, 100\}$.
   - Анализ сеточной сходимости по норме $E_\infty = \max_j |u_j - u(x_j, t)|$ и оценка порядка сходимости.
   - Физико-математическое объяснение опережающего затухания высокочастотных гармоник.
   
2:
   - Реализация схемы Кранка — Николсона.
   - Численное моделирование теплопереноса при разрывном начальном профиле на сетке $N = 50$.
   - Исследование поведения неявных схем при умеренных ($r = 0.4$) и крупных шагах ($r = 2$, $r = 10$).
   - Детальный анализ первых временных слоёв в окрестности точек разрыва $x = 0.4$ и $x = 0.6$.
   - Динамика экстремумов $\min_j u_j^n$ и $\max_j u_j^n$, проверка дискретного принципа максимума.
   - Сопоставление решений с эталонным расчётом на мелком шаге и ответ на вопрос о взаимосвязи безусловной устойчивости, точности и монотонности.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Настройки оформления графиков в строгом академическом стиле
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Единая палитра
BLUE   = "#0072B2"
ORANGE = "#D55E00"
GREEN  = "#009E73"
PURPLE = "#8B4BA5"
RED    = "#D52A2A"
GRAY   = "#555555"

def table(headers, rows):
    # Вывод форматированной таблицы в формате Markdown
    header_line = "| " + " | ".join(headers) + " |"
    sep_line = "| " + " | ".join("---:" if i > 0 else ":---" for i in range(len(headers))) + " |"
    row_lines = ["| " + " | ".join(str(val) for val in r) + " |" for r in rows]
    display(Markdown("\n".join([header_line, sep_line] + row_lines)))


## 1. Разностные схемы для уравнения теплопроводности

Введём равномерную пространственно-временную сетку:
$$
x_j = j \Delta x, \quad j = 0, 1, \dots, N, \quad \Delta x = \frac{1}{N},
$$
$$
t_n = n \Delta t, \quad n = 0, 1, \dots, \quad \Delta t = r \Delta x^2.
$$
Здесь $r = \frac{\Delta t}{\Delta x^2}$ — сеточный сеточный параметр Фурье (число Куранта для параболических уравнений). Обозначим $u_j^n \approx u(x_j, t_n)$, а вторую разностную производную — $\delta_x^2 u_j^n = u_{j+1}^n - 2u_j^n + u_{j-1}^n$.

### 1.1. Явная схема FTCS (Forward Time, Central Space)
$$
\frac{u_j^{n+1} - u_j^n}{\Delta t} = \frac{u_{j+1}^n - 2u_j^n + u_{j-1}^n}{\Delta x^2} \implies u_j^{n+1} = u_j^n + r \delta_x^2 u_j^n.
$$
- Погрешность аппроксимации: $O(\Delta t + \Delta x^2) = O(\Delta x^2)$ при фиксированном $r$.
- Устойчивость по Нейману: схема условно устойчива при $r \le \frac{1}{2}$. При $r > \frac{1}{2}$ высокочастотные компоненты экспоненциально нарастают.

### 1.2. Неявная схема Эйлера (Backward Euler)
$$
\frac{u_j^{n+1} - u_j^n}{\Delta t} = \frac{\delta_x^2 u_j^{n+1}}{\Delta x^2} \implies -r u_{j-1}^{n+1} + (1 + 2r) u_j^{n+1} - r u_{j+1}^{n+1} = u_j^n.
$$
- Погрешность аппроксимации: $O(\Delta t + \Delta x^2)$.
- Устойчивость: схема безусловно устойчива (A-устойчива и L-устойчива).
- Монотонность: матрица системы является $M$-матрицей при любых $r > 0$, что гарантирует строгое выполнение дискретного принципа максимума и отсутствие паразитных осцилляций.

### 1.3. Схема Кранка — Николсона (Crank — Nicolson)
Симметричная полунеявная схема с осреднением по времени на полуцелом слое $t_{n+1/2}$:
$$
\frac{u_j^{n+1} - u_j^n}{\Delta t} = \frac{1}{2}\left[\frac{\delta_x^2 u_j^{n+1}}{\Delta x^2} + \frac{\delta_x^2 u_j^n}{\Delta x^2}\right],
$$
что приводит к СЛАУ относительно вектора на новом временном слое:
$$
-\frac{r}{2} u_{j-1}^{n+1} + (1 + r) u_j^{n+1} - \frac{r}{2} u_{j+1}^{n+1} = \frac{r}{2} u_{j-1}^n + (1 - r) u_j^n + \frac{r}{2} u_{j+1}^n.
$$
- Погрешность аппроксимации: $O(\Delta t^2 + \Delta x^2)$. При $r = \text{const}$ временной вклад $O(\Delta t^2) = O(\Delta x^4)$, поэтому ведущей ошибкой остаётся $O(\Delta x^2)$.
- Устойчивость: безусловно устойчива по Нейману (A-устойчива), однако **не является L-устойчивой**.

### 1.4. Спектральные коэффициенты усиления (анализ фон Неймана)
Подставляя гармонику $u_j^n = G^n e^{i k x_j}$ с волновым числом $k \in [1, N-1]$ и фазовым параметром $\theta_k = \frac{k \pi}{N} \in (0, \pi)$, получаем фактор усиления $G$:
$$
\mu_k = 4 \sin^2\left(\frac{\theta_k}{2}\right) \in (0, 4),
$$
$$
G_{\text{FTCS}} = 1 - r \mu_k, \qquad G_{\text{IE}} = \frac{1}{1 + r \mu_k}, \qquad G_{\text{CN}} = \frac{1 - \frac{r \mu_k}{2}}{1 + \frac{r \mu_k}{2}}.
$$

При больших шагах $r \mu_k \gg 1$ для старших пространственных частот:
- $G_{\text{IE}} \to 0^+$ (быстрое сглаживание, правильное качественное поведение);
- $G_{\text{CN}} \to -1$ (гармоники затухают крайне медленно и каждый временной шаг меняют знак, порождая осцилляции шестерёнчатого типа).

In [ ]:
def solve_tridiagonal(a, b, c, d):
    # Классический алгоритм прогонки (метод Томаса) для трёхдиагональной СЛАУ:
    # a[i]*x[i-1] + b[i]*x[i] + c[i]*x[i+1] = d[i],  i = 0, ..., M-1.
    m = len(b)
    cp = np.empty(m - 1, dtype=np.float64)
    dp = np.empty(m, dtype=np.float64)
    
    cp[0] = c[0] / b[0]
    dp[0] = d[0] / b[0]
    for i in range(1, m - 1):
        denom = b[i] - a[i] * cp[i - 1]
        cp[i] = c[i] / denom
        dp[i] = (d[i] - a[i] * dp[i - 1]) / denom
        
    denom_last = b[-1] - a[-1] * cp[-1]
    dp[-1] = (d[-1] - a[-1] * dp[-2]) / denom_last
    
    x = np.empty(m, dtype=np.float64)
    x[-1] = dp[-1]
    for i in range(m - 2, -1, -1):
        x[i] = dp[i] - cp[i] * x[i + 1]
    return x


def heat_solver(N, r, t_end, method="ftcs", u0_func=None, store_history=False):
    # Универсальный решатель уравнения теплопроводности u_t = u_xx.
    # N             : число пространственных интервалов (узлов N+1)
    # r             : сеточное отношение dt / dx^2
    # t_end         : целевой момент времени
    # method        : 'ftcs', 'ie' (неявный Эйлер), 'cn' (Кранк-Николсон)
    # u0_func       : функция начального профиля u0(x)
    # store_history : если True, сохраняется траектория слоёв и экстремумов
    dx = 1.0 / N
    x = np.linspace(0.0, 1.0, N + 1)
    dt_base = r * (dx ** 2)
    
    u = u0_func(x).astype(np.float64)
    u[0] = 0.0
    u[-1] = 0.0
    
    t = 0.0
    t_hist = [0.0]
    u_hist = [u.copy()]
    min_hist = [np.min(u[1:-1])]
    max_hist = [np.max(u[1:-1])]
    
    m = N - 1  # число внутренних узлов
    
    while t < t_end - 1e-15:
        # Корректировка последнего шага для точного попадания в t_end
        dt = min(dt_base, t_end - t)
        curr_r = dt / (dx ** 2)
        
        u_prev = u.copy()
        interior = u_prev[1:-1]
        laplacian = u_prev[:-2] - 2.0 * interior + u_prev[2:]
        
        if method == "ftcs":
            u[1:-1] = interior + curr_r * laplacian
            
        elif method == "ie":
            # (1 + 2r)*u_j^{n+1} - r*(u_{j-1}^{n+1} + u_{j+1}^{n+1}) = u_j^n
            a = np.full(m, -curr_r)
            b = np.full(m, 1.0 + 2.0 * curr_r)
            c = np.full(m, -curr_r)
            rhs = interior.copy()
            u[1:-1] = solve_tridiagonal(a, b, c, rhs)
            
        elif method == "cn":
            # (1 + r)*u_j^{n+1} - (r/2)*(u_{j-1}^{n+1} + u_{j+1}^{n+1}) =
            # = (1 - r)*u_j^n + (r/2)*(u_{j-1}^n + u_{j+1}^n)
            half_r = 0.5 * curr_r
            a = np.full(m, -half_r)
            b = np.full(m, 1.0 + 2.0 * half_r)
            c = np.full(m, -half_r)
            rhs = interior + half_r * laplacian
            u[1:-1] = solve_tridiagonal(a, b, c, rhs)
        else:
            raise ValueError(f"Неизвестный метод: {method}")
            
        u[0] = 0.0
        u[-1] = 0.0
        t += dt
        
        if store_history:
            t_hist.append(t)
            u_hist.append(u.copy())
            min_hist.append(np.min(u[1:-1]))
            max_hist.append(np.max(u[1:-1]))
            
    if store_history:
        return x, np.array(t_hist), np.array(u_hist), np.array(min_hist), np.array(max_hist)
    return x, u


def u_exact_smooth(x, t):
    # Точное решение для гладкого двухмодового начального профиля
    return (np.exp(-np.pi**2 * t) * np.sin(np.pi * x) +
            0.3 * np.exp(-25.0 * np.pi**2 * t) * np.sin(5.0 * np.pi * x))


def u0_smooth(x):
    return np.sin(np.pi * x) + 0.3 * np.sin(5.0 * np.pi * x)


def u0_step(x):
    res = np.zeros_like(x)
    mask = (x >= 0.4 - 1e-12) & (x <= 0.6 + 1e-12)
    res[mask] = 1.0
    return res


---
## 2. Гладкое начальное распределение

Рассматривается начальное распределение температуры:
$$
u(x, 0) = \sin(\pi x) + 0.3 \sin(5\pi x).
$$
Фиксируется сеточное отношение $r = \frac{\Delta t}{\Delta x^2} = 0.4$, что лежит в области устойчивости явной схемы FTCS ($r \le 0.5$). При совместном измельчении шагов по сеткам $N \in \{25, 50, 100\}$ пространственный шаг $\Delta x = 1/N$ уменьшается в 2 раза, а временной шаг $\Delta t = r \Delta x^2$ — в 4 раза.

Сравнение проводится в три характерных момента времени:
- $t = 0.001$: начальная стадия, обе гармоники вносят заметный вклад ($e^{-\pi^2 t} \approx 0.990$, $e^{-25\pi^2 t} \approx 0.781$);
- $t = 0.01$: промежуточная стадия, 5-я гармоника затухает на порядок ($e^{-25\pi^2 t} \approx 0.0848$), профиль сглаживается;
- $t = 0.1$: квазистационарная стадия, 5-я гармоника практически исчезает ($e^{-25\pi^2 t} \approx 1.88 \cdot 10^{-11}$), остаётся только главная мода.

In [ ]:
times_green = [0.001, 0.01, 0.1]
Ns_green = [25, 50, 100]
r_green = 0.4

results_green = {}

for method in ["ftcs", "ie"]:
    for N in Ns_green:
        for t_val in times_green:
            x, u_num = heat_solver(N, r_green, t_val, method=method, u0_func=u0_smooth)
            results_green[(method, N, t_val)] = (x, u_num)

fig, axes = plt.subplots(len(times_green), len(Ns_green), figsize=(14, 10), sharex=True, layout="constrained")
x_dense = np.linspace(0.0, 1.0, 500)

for row_idx, t_val in enumerate(times_green):
    for col_idx, N in enumerate(Ns_green):
        ax = axes[row_idx, col_idx]
        
        # Точное решение
        ax.plot(x_dense, u_exact_smooth(x_dense, t_val), color="black", lw=1.5, ls="--", label="Точное", zorder=4)
        
        # FTCS
        x_f, u_f = results_green[("ftcs", N, t_val)]
        ax.plot(x_f, u_f, color=BLUE, marker="o", ms=3, lw=1.2, label="FTCS", alpha=0.85)
        
        # Неявный Эйлер
        x_ie, u_ie = results_green[("ie", N, t_val)]
        ax.plot(x_ie, u_ie, color=ORANGE, marker="s", ms=2.5, lw=1.2, label="Неявный Эйлер", alpha=0.85)
        
        if row_idx == 0:
            ax.set_title(f"$N = {N}$ ($\\Delta x = {1/N:.3f}$)", fontsize=11, fontweight="bold")
        if col_idx == 0:
            ax.set_ylabel(f"$t = {t_val}$\n$u(x, t)$", fontsize=11)
        if row_idx == len(times_green) - 1:
            ax.set_xlabel("$x$")
        if row_idx == 0 and col_idx == len(Ns_green) - 1:
            ax.legend(loc="upper right", fontsize=8.5)

fig.suptitle("Сравнение профилей температуры $u(x, t)$ при $r = 0.4$", fontsize=13, y=1.02)
plt.show()


### 2.1. Таблица погрешностей и оценка сеточного порядка сходимости

Максимальная абсолютная ошибка решения на сетке вычисляется по Чебышёвской норме:
$$
E_\infty(N) = \max_{0 \le j \le N} |u_j - u(x_j, t)|.
$$
Поскольку при удвоении числа узлов ($N \to 2N$) шаг $\Delta x$ уменьшается в 2 раза, эффективный порядок сходимости определяется логарифмическим отношением погрешностей:
$$
p = \log_2\left(\frac{E_\infty(N)}{E_\infty(2N)}\right).
$$
Так как шаг по времени согласован как $\Delta t = r \Delta x^2$, локальная ошибка обеих схем $O(\Delta t + \Delta x^2) = O(\Delta x^2)$, поэтому теоретический порядок сходимости равен $p = 2$.

In [ ]:
table_data = []

for t_val in times_green:
    for method, method_name in [("ftcs", "FTCS"), ("ie", "Неявный Эйлер")]:
        prev_err = None
        for N in Ns_green:
            x, u_num = results_green[(method, N, t_val)]
            u_ex = u_exact_smooth(x, t_val)
            err = np.max(np.abs(u_num - u_ex))
            
            if prev_err is not None:
                order = np.log2(prev_err / err)
                order_str = f"{order:.2f}"
            else:
                order_str = "—"
            prev_err = err
            
            dx_val = 1.0 / N
            dt_val = r_green * (dx_val ** 2)
            table_data.append([
                f"{t_val}",
                method_name,
                N,
                f"{dx_val:.4f}",
                f"{dt_val:.3e}",
                f"{err:.4e}",
                order_str
            ])

headers_green = ["$t$", "Метод", "$N$", r"$\Delta x$", r"$\Delta t$", r"$E_\infty$", "Порядок $p$"]
table(headers_green, table_data)


### 2.2. Физико-математический анализ: почему мелкие детали температурного профиля исчезают быстрее крупных?

**Математическое обоснование:**
Решение однородного уравнения теплопроводности представляется в виде ряда Фурье:
$$
u(x, t) = \sum_{k=1}^\infty A_k e^{-\lambda_k t} \sin(k \pi x), \qquad \lambda_k = (k \pi)^2.
$$
Показатель затухания $\lambda_k$ (собственное значение дифференциального оператора $-\frac{\partial^2}{\partial x^2}$) квадратично зависит от волнового числа $k$:
$$
\lambda_k \propto k^2.
$$
В рассмотренном начальном профиле присутствуют две гармоники:
- Первая мода ($k = 1$, крупный масштаб $\ell_1 = 1$): $\lambda_1 = \pi^2 \approx 9.87$ с характеристическим временем релаксации $\tau_1 = \frac{1}{\lambda_1} \approx 0.101$.
- Пятая мода ($k = 5$, мелкий масштаб $\ell_5 = 1/5$): $\lambda_5 = 25\pi^2 \approx 246.74$ с временем релаксации $\tau_5 = \frac{1}{\lambda_5} \approx 0.00405$.

Таким образом, 5-я мода затухает ровно в $5^2 = 25$ раз быстрее, чем 1-я. Уже к моменту $t = 0.01$ амплитуда пятой гармоники падает в $e^{2.467} \approx 11.8$ раз, а к $t = 0.1$ её относительный вклад составляет $\sim 10^{-11}$, и температурный профиль приобретает форму фундаментальной синусоиды.

**Физический механизм:**
Согласно закону Фурье, плотность теплового потока пропорциональна пространственному градиенту температуры:
$$
q = -\kappa \frac{\partial u}{\partial x}.
$$
Для пространственного возмущения с амплитудой $A$ и пространственным масштабом $\lambda$ градиент температуры масштабируется как $\frac{\partial u}{\partial x} \sim \frac{A}{\lambda}$, а дивергенция потока (скорость изменения температуры) — как:
$$
\frac{\partial u}{\partial t} = \frac{\partial^2 u}{\partial x^2} \sim -\frac{A}{\lambda^2}.
$$
При уменьшении характерного пространственного размера возмущения в 5 раз расстояние между областями нагрева и охлаждения уменьшается в 5 раз, градиент температуры возрастает в 5 раз, а диффузионное перераспределение тепла ускоряется в 25 раз.

---
## 3. Разрывное начальное распределение и крупные шаги

Рассматривается стержень с нагретым центральным участком:
$$
u(x, 0) = \begin{cases} 1, & 0.4 \le x \le 0.6, \\ 0, & x < 0.4 \;\text{или}\; x > 0.6. \end{cases}
$$
В точках $x = 0.4$ и $x = 0.6$ функция имеет разрывы первого рода. Разложение в ряд Фурье такого профиля:
$$
u(x, 0) = \sum_{k=1}^\infty \frac{2}{k\pi}\left[\cos(0.4 k\pi) - \cos(0.6 k\pi)\right] \sin(k\pi x)
$$
спадает со скоростью всего $O(1/k)$, возбуждая сколь угодно высокие пространственные частоты.

Исследование проводится на фиксированной сетке $N = 50$ ($\Delta x = 0.02$), где границы нагретой зоны $x = 0.4$ ($j = 20$) и $x = 0.6$ ($j = 30$) точно попадают в сеточные узлы. Рассматриваются три режима:
- $r = 0.4$ (мелкий шаг $\Delta t = 1.6 \cdot 10^{-4}$): применимы все три схемы (FTCS, IE, CN);
- $r = 2.0$ (умеренный шаг $\Delta t = 8.0 \cdot 10^{-4}$): только неявные схемы (IE, CN);
- $r = 10.0$ (крупный шаг $\Delta t = 4.0 \cdot 10^{-3}$): только неявные схемы (IE, CN).

> Согласно постановке задачи, вычисленные отрицательные значения температуры **не обрезаются**, что позволяет беспристрастно исследовать вычислительные артефакты схем.

In [ ]:
N_yellow = 50
dx_y = 1.0 / N_yellow
r_cases = [0.4, 2.0, 10.0]

# Расчёт истории для всех вариантов
sim_data = {}

# r = 0.4 (FTCS, IE, CN)
for m in ["ftcs", "ie", "cn"]:
    sim_data[(m, 0.4)] = heat_solver(N_yellow, 0.4, t_end=0.03, method=m, 
                                     u0_func=u0_step, store_history=True)

# r = 2.0 и r = 10.0 (только IE и CN)
for r_val in [2.0, 10.0]:
    for m in ["ie", "cn"]:
        sim_data[(m, r_val)] = heat_solver(N_yellow, r_val, t_end=0.03, method=m, 
                                         u0_func=u0_step, store_history=True)


### 3.1. Первые временные слои и увеличенные фрагменты у границ разрыва

Построим профили температуры на первых временных слоях ($n = 0, 1, 2, 3, 5$) для $r = 0.4$, $r = 2.0$ и $r = 10.0$, сопроводив их детальным увеличением зон разрыва при $x \in [0.35, 0.65]$.

In [ ]:
fig = plt.figure(figsize=(15, 12), layout="constrained")
gs = fig.add_gridspec(3, 2, width_ratios=[1.2, 1.0])

methods_labels = {"ftcs": "FTCS", "ie": "Неявный Эйлер", "cn": "Кранк — Николсон"}
palette_layers = [GRAY, BLUE, GREEN, ORANGE, RED]

# Панель 1: r = 0.4 (сравнение всех трёх методов на шаге n=1)
ax1 = fig.add_subplot(gs[0, 0])
x_grid = sim_data[("ie", 0.4)][0]
ax1.plot(x_grid, sim_data[("ie", 0.4)][2][0], color=GRAY, ls="--", lw=1.2, label="$n=0$ (нач. профиль)")
ax1.plot(x_grid, sim_data[("ftcs", 0.4)][2][1], color=BLUE, lw=1.4, marker="o", ms=3, label="FTCS ($n=1$)")
ax1.plot(x_grid, sim_data[("ie", 0.4)][2][1], color=ORANGE, lw=1.4, marker="s", ms=3, label="Неявный Эйлер ($n=1$)")
ax1.plot(x_grid, sim_data[("cn", 0.4)][2][1], color=GREEN, lw=1.4, marker="^", ms=3, label="Кранк — Николсон ($n=1$)")
ax1.set_title(r"Первый слой ($n=1$, $\Delta t = 1.6\cdot 10^{-4}$) при $r = 0.4$", fontsize=11, fontweight="bold")
ax1.set_xlabel("$x$")
ax1.set_ylabel("$u(x, t)$")
ax1.legend(loc="upper right", fontsize=8.5)

# Панель 1 (Zoom): Увеличенный фрагмент у границы x=0.4
ax1_z = fig.add_subplot(gs[0, 1])
ax1_z.plot(x_grid, sim_data[("ie", 0.4)][2][0], color=GRAY, ls="--", lw=1.2, label="$n=0$")
ax1_z.plot(x_grid, sim_data[("ftcs", 0.4)][2][1], color=BLUE, marker="o", ms=4, lw=1.4, label="FTCS")
ax1_z.plot(x_grid, sim_data[("ie", 0.4)][2][1], color=ORANGE, marker="s", ms=4, lw=1.4, label="Неявный Эйлер")
ax1_z.plot(x_grid, sim_data[("cn", 0.4)][2][1], color=GREEN, marker="^", ms=4, lw=1.4, label="Кранк — Николсон")
ax1_z.set_xlim(0.32, 0.48)
ax1_z.set_title(r"Увеличение границы $x = 0.4$ ($r = 0.4$)", fontsize=11, fontweight="bold")
ax1_z.set_xlabel("$x$")
ax1_z.legend(loc="upper left", fontsize=8.5)

# Панель 2: r = 2.0 (первые слои CN vs IE)
ax2 = fig.add_subplot(gs[1, 0])
for step_idx, step_n in enumerate([0, 1, 2, 3, 5]):
    u_ie_layer = sim_data[("ie", 2.0)][2][step_n]
    u_cn_layer = sim_data[("cn", 2.0)][2][step_n]
    col = palette_layers[step_idx]
    ax2.plot(x_grid, u_cn_layer, color=col, lw=1.3, label=f"CN $n={step_n}$")
    ax2.plot(x_grid, u_ie_layer, color=col, ls=":", lw=1.3, label=f"IE $n={step_n}$" if step_idx==1 else "")
ax2.axhline(0.0, color="black", lw=0.8, ls=":")
ax2.axhline(1.0, color="black", lw=0.8, ls=":")
ax2.set_title(r"Слои $n \in \{0, 1, 2, 3, 5\}$ при $r = 2.0$ (сплошная: CN, пунктир: IE)", fontsize=11, fontweight="bold")
ax2.set_xlabel("$x$")
ax2.set_ylabel("$u(x, t)$")
ax2.legend(loc="upper right", fontsize=8, ncol=2)

# Панель 2 (Zoom): r = 2.0 окрестность x=0.4
ax2_z = fig.add_subplot(gs[1, 1])
ax2_z.plot(x_grid, sim_data[("cn", 2.0)][2][1], color=RED, marker="o", ms=4, lw=1.4, label="CN ($n=1$)")
ax2_z.plot(x_grid, sim_data[("ie", 2.0)][2][1], color=BLUE, marker="s", ms=4, lw=1.4, label="IE ($n=1$)")
ax2_z.plot(x_grid, sim_data[("cn", 2.0)][2][0], color=GRAY, ls="--", label="$n=0$")
ax2_z.axhline(0.0, color="black", lw=0.8, ls=":")
ax2_z.set_xlim(0.30, 0.50)
ax2_z.set_ylim(-0.02, 1.05)
ax2_z.set_title(r"Увеличение границы $x = 0.4$ при $r = 2.0$ (шаг $n=1$)", fontsize=11, fontweight="bold")
ax2_z.set_xlabel("$x$")
ax2_z.legend(loc="upper left", fontsize=8.5)

# Панель 3: r = 10.0 (первые слои CN vs IE)
ax3 = fig.add_subplot(gs[2, 0])
for step_idx, step_n in enumerate([0, 1, 2, 3, 5]):
    u_ie_layer = sim_data[("ie", 10.0)][2][step_n]
    u_cn_layer = sim_data[("cn", 10.0)][2][step_n]
    col = palette_layers[step_idx]
    ax3.plot(x_grid, u_cn_layer, color=col, lw=1.3, label=f"CN $n={step_n}$")
    ax3.plot(x_grid, u_ie_layer, color=col, ls=":", lw=1.3, label=f"IE $n={step_n}$" if step_idx==1 else "")
ax3.axhline(0.0, color="black", lw=0.8, ls=":")
ax3.axhline(1.0, color="black", lw=0.8, ls=":")
ax3.set_title(r"Слои $n \in \{0, 1, 2, 3, 5\}$ при $r = 10.0$ (сплошная: CN, пунктир: IE)", fontsize=11, fontweight="bold")
ax3.set_xlabel("$x$")
ax3.set_ylabel("$u(x, t)$")
ax3.legend(loc="upper right", fontsize=8, ncol=2)

# Панель 3 (Zoom): r = 10.0 окрестность x=0.4
ax3_z = fig.add_subplot(gs[2, 1])
ax3_z.plot(x_grid, sim_data[("cn", 10.0)][2][1], color=RED, marker="o", ms=4, lw=1.4, label="CN ($n=1$)")
ax3_z.plot(x_grid, sim_data[("ie", 10.0)][2][1], color=BLUE, marker="s", ms=4, lw=1.4, label="IE ($n=1$)")
ax3_z.plot(x_grid, sim_data[("cn", 10.0)][2][0], color=GRAY, ls="--", label="$n=0$")
ax3_z.axhline(0.0, color="black", lw=0.8, ls=":")
ax3_z.set_xlim(0.30, 0.50)
ax3_z.set_ylim(-0.02, 1.05)
ax3_z.set_title(r"Увеличение границы $x = 0.4$ при $r = 10.0$ (шаг $n=1$)", fontsize=11, fontweight="bold")
ax3_z.set_xlabel("$x$")
ax3_z.legend(loc="upper left", fontsize=8.5)

fig.suptitle("Эволюция профиля температуры и паразитные осцилляции схем", fontsize=13, y=1.02)
plt.show()


### 3.2. Динамика экстремумов решения и локальная монотонность

Для непрерывного уравнения теплопроводности без источников верен **принцип максимума**:
$$
0 = \min_x u(x, 0) \le u(x, t) \le \max_x u(x, 0) = 1 \qquad \forall t \ge 0.
$$
Физически температура не может опуститься ниже минимальной начальной ($u < 0$) или превысить максимальную ($u > 1$). Кроме того, диффузионный процесс обладает свойством монотонности: тепло самопроизвольно перетекает исключительно от более нагретых областей к менее нагретым, и возникновение новых локальных экстремумов или пространственных осцилляций физически невозможно.

Построим графики глобального минимума $\min_j u_j(t)$ и максимума $\max_j u_j(t)$ во времени для всех исследуемых режимов.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")

# График минимума
ax_min = axes[0]
# r = 0.4
ax_min.plot(sim_data[("ftcs", 0.4)][1], sim_data[("ftcs", 0.4)][3], color=BLUE, lw=1.2, ls="-", label="FTCS ($r=0.4$)")
ax_min.plot(sim_data[("ie", 0.4)][1], sim_data[("ie", 0.4)][3], color=ORANGE, lw=1.2, ls="--", label="IE ($r=0.4$)")
ax_min.plot(sim_data[("cn", 0.4)][1], sim_data[("cn", 0.4)][3], color=GREEN, lw=1.2, ls="-.", label="CN ($r=0.4$)")
# r = 2.0
ax_min.plot(sim_data[("ie", 2.0)][1], sim_data[("ie", 2.0)][3], color="#E69F00", lw=1.5, ls="--", label="IE ($r=2.0$)")
ax_min.plot(sim_data[("cn", 2.0)][1], sim_data[("cn", 2.0)][3], color="#009E73", lw=1.5, ls="-.", label="CN ($r=2.0$)")
# r = 10.0
ax_min.plot(sim_data[("ie", 10.0)][1], sim_data[("ie", 10.0)][3], color=PURPLE, lw=1.5, ls="--", label="IE ($r=10.0$)")
ax_min.plot(sim_data[("cn", 10.0)][1], sim_data[("cn", 10.0)][3], color=RED, lw=1.8, ls="-", label="CN ($r=10.0$)")

ax_min.axhline(0.0, color="black", lw=1.0, ls=":", label=r"Физическая граница $u \geq 0$")
ax_min.set_title(r"Динамика глобального минимума $\min_j u_j(t)$", fontsize=11, fontweight="bold")
ax_min.set_xlabel("Время $t$")
ax_min.set_ylabel(r"$\min_j u_j$")
ax_min.set_xlim(0, 0.02)
ax_min.set_ylim(-0.02, 0.15)
ax_min.legend(loc="lower right", fontsize=8.5)

# График максимума
ax_max = axes[1]
# r = 0.4
ax_max.plot(sim_data[("ftcs", 0.4)][1], sim_data[("ftcs", 0.4)][4], color=BLUE, lw=1.2, ls="-", label="FTCS ($r=0.4$)")
ax_max.plot(sim_data[("ie", 0.4)][1], sim_data[("ie", 0.4)][4], color=ORANGE, lw=1.2, ls="--", label="IE ($r=0.4$)")
ax_max.plot(sim_data[("cn", 0.4)][1], sim_data[("cn", 0.4)][4], color=GREEN, lw=1.2, ls="-.", label="CN ($r=0.4$)")
# r = 2.0
ax_max.plot(sim_data[("ie", 2.0)][1], sim_data[("ie", 2.0)][4], color="#E69F00", lw=1.5, ls="--", label="IE ($r=2.0$)")
ax_max.plot(sim_data[("cn", 2.0)][1], sim_data[("cn", 2.0)][4], color="#009E73", lw=1.5, ls="-.", label="CN ($r=2.0$)")
# r = 10.0
ax_max.plot(sim_data[("ie", 10.0)][1], sim_data[("ie", 10.0)][4], color=PURPLE, lw=1.5, ls="--", label="IE ($r=10.0$)")
ax_max.plot(sim_data[("cn", 10.0)][1], sim_data[("cn", 10.0)][4], color=RED, lw=1.8, ls="-", label="CN ($r=10.0$)")

ax_max.axhline(1.0, color="black", lw=1.0, ls=":", label=r"Физическая граница $u \leq 1$")
ax_max.set_title(r"Динамика глобального максимума $\max_j u_j(t)$", fontsize=11, fontweight="bold")
ax_max.set_xlabel("Время $t$")
ax_max.set_ylabel(r"$\max_j u_j$")
ax_max.set_xlim(0, 0.02)
ax_max.legend(loc="upper right", fontsize=8.5)

plt.show()


### 3.3. Сопоставление с эталонным расчётом на малом временном шаге

В качестве эталонного численного решения выберем расчёт по схеме FTCS на той же сетке $N = 50$ с малым шагом $r = 0.4$ ($\Delta t = 1.6 \cdot 10^{-4}$), гарантирующим высокую точность аппроксимации по времени и полное отсутствие осцилляций.

Сопоставим профили при двух фиксированных моментах времени:
1. $t = 0.004$ (ранний момент: 25 шагов FTCS, 5 шагов схемы при $r = 2$, ровно 1 шаг при $r = 10$);
2. $t = 0.020$ (поздний момент: выраженная диффузия, 125 шагов FTCS, 25 шагов при $r=2$, 5 шагов при $r = 10$).

In [ ]:
compare_times = [0.004, 0.020]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), sharey=True, layout="constrained")

for idx, t_comp in enumerate(compare_times):
    ax = axes[idx]
    
    # Эталон FTCS r=0.4
    x_ref, u_ref = heat_solver(N_yellow, 0.4, t_end=t_comp, method="ftcs", u0_func=u0_step)
    ax.plot(x_ref, u_ref, color="black", lw=2.5, ls="-", label="Эталон (FTCS, $r=0.4$)", zorder=5)
    
    # Неявный Эйлер
    for r_val, col, ls in [(0.4, ORANGE, ":"), (2.0, "#E69F00", "--"), (10.0, PURPLE, "-.")]:
        x_m, u_m = heat_solver(N_yellow, r_val, t_end=t_comp, method="ie", u0_func=u0_step)
        ax.plot(x_m, u_m, color=col, lw=1.3, ls=ls, label=f"IE ($r={r_val}$)")
        
    # Кранк-Николсон
    for r_val, col, ls in [(0.4, GREEN, ":"), (2.0, "#009E73", "--"), (10.0, RED, "-")]:
        x_m, u_m = heat_solver(N_yellow, r_val, t_end=t_comp, method="cn", u0_func=u0_step)
        ax.plot(x_m, u_m, color=col, lw=1.4, ls=ls, label=f"CN ($r={r_val}$)")
        
    ax.axhline(0.0, color="gray", lw=0.8, ls=":")
    ax.set_title(f"Профили при $t = {t_comp}$", fontsize=11, fontweight="bold")
    ax.set_xlabel("$x$")
    if idx == 0:
        ax.set_ylabel("$u(x, t)$")
    ax.legend(loc="upper right", fontsize=8.0)

fig.suptitle("Сопоставление неявных схем с эталонным расчётом на мелком шаге", fontsize=13, y=1.02)
plt.show()


### 3.4. Аналитический ответ: гарантирует ли безусловная устойчивость точность и отсутствие осцилляций?

**Ответ:** **Нет, не гарантирует.** Безусловная устойчивость по Нейману гарантирует лишь ограниченность нормы оператора шага во времени ($\|u^n\| \le C \|u^0\|$), что исключает неограниченный взрывной рост решения при любых шагах $\Delta t$. Однако она не влечёт за собой ни высокой точности, ни физической адекватности (монотонности) численного профиля.

#### 1. Механизм осцилляций схемы Кранка — Николсона ($L$-неустойчивость)
Спектральный фактор усиления схемы Кранка — Николсона для гармоники с фазой $\theta = k \pi \Delta x$ имеет вид:
$$
G_{\text{CN}}(\theta) = \frac{1 - \frac{r \mu}{2}}{1 + \frac{r \mu}{2}}, \qquad \mu = 4 \sin^2\left(\frac{\theta}{2}\right) \in (0, 4).
$$
- Для наивысшей пространственной гармоники сетки ($\theta = \pi$, $\mu = 4$):
  $$
  G_{\text{CN}}(\pi) = \frac{1 - 2r}{1 + 2r}.
  $$
- Условие монотонности (неотрицательности множителя $G \ge 0$) требует:
  $$
  1 - 2r \ge 0 \implies r \le 0.5.
  $$
- При $r = 2.0$: $G(\pi) = \frac{1 - 4}{1 + 4} = -0.6 < 0$.
- При $r = 10.0$: $G(\pi) = \frac{1 - 20}{1 + 20} \approx -0.905 < 0$.

Так как $|G| < 1$, схема остаётся формально устойчивой по Нейману. Однако **отрицательный знак фактора усиления старшей гармоники** ($G < 0$) означает, что амплитуда этой высокочастотной составляющей на каждом временном шаге меняет знак:
$$
G^n = (-1)^n |G|^n.
$$
При $r = 10$ затухание составляет всего $\approx 9.5\%$ за шаг ($|G| \approx 0.905$), поэтому паразитные высокочастотные колебания затухают крайне медленно и порождают длительно наблюдаемую пространственно-временную «шестерёнку».

**Соотношение со средней температурой и минимумом:**
Полное численное решение складывается из гладкого положительного фона (средней температуры стержня $\bar{u} \approx 0.216$, определяемой низкими гармониками) и знакопеременных высокочастотных компонент. Поэтому абсолютная температура $u(x, t)$ **не опускается ниже нуля**: глобальный минимум на краях стержня остаётся строго равным нулю ($\min_j u_j = 0$), что полностью согласуется с графиком минимума. 

Численный артефакт схемы Кранка — Николсона проявляется в **грубом нарушении локальной монотонности и инверсии теплового потока**:
- На шаге $n = 1$ при $r = 10$ в холодном узле $x = 0.38$ температура скачкообразно возрастает до $0.78$, а в соседнем горячем узле $x = 0.40$ проседает до $0.21$.
- На шаге $n = 2$ происходит обратная переполюсовка: в $x = 0.38$ температура опускается до $0.18$, а в $x = 0.40$ вновь возрастает до $0.75$.
Таким образом, тепло фиктивно «перетекает» из холодной области в горячую и обратно с периодом $2\Delta t$, что противоречит второму началу термодинамики.

#### 2. Чрезмерная диссипация неявной схемы Эйлера
Неявная схема Эйлера имеет множитель:
$$
G_{\text{IE}}(\theta) = \frac{1}{1 + r \mu} > 0 \quad \forall r > 0.
$$
Она монотонна и $L$-устойчива ($\lim_{r\mu \to \infty} G_{\text{IE}} = 0$): множитель строго положителен для всех частот, поэтому осцилляции и переполюсовка исключены. Однако при крупных шагах $r = 10$ из-за погрешности $O(\Delta t)$ схема вносит чрезмерную схемную гипердиффузию, мгновенно размывая фронт разрыва существенно сильнее истинной физической теплопроводности.

#### 3. Практический вывод
Безусловная устойчивость схемы предотвращает расходимость, но **не гарантирует ни точности, ни монотонности**. Для расчёта разрывных задач схемой Кранка — Николсона шаг должен выбираться из условия монотонности $r \le 0.5$ либо на первых шагах должно применяться специальное демпфирование (например, несколько начальных полушагов по схеме Раannacher / неявного Эйлера).

---
## Выводы

1. **Сходимость на гладком решении:**
   - На гладком начальном условии $u(x, 0) = \sin(\pi x) + 0.3\sin(5\pi x)$ при фиксированном отношении $r = \Delta t / \Delta x^2 = 0.4$ обе схемы (FTCS и неявный Эйлер) продемонстрировали устойчивую сходимость со вторым порядком $p \approx 2.0$ по пространственной норме $E_\infty$.
   - Погрешности FTCS и неявного Эйлера имеют противоположный знак временной аппроксимации, но по абсолютной величине близки благодаря доминированию пространственной аппроксимации $O(\Delta x^2)$.
   - Экспериментально подтверждено опережающее затухание 5-й гармоники: её декремент затухания $\lambda_5 = 25\pi^2$ ровно в 25 раз превышает декремент первой моды $\lambda_1 = \pi^2$, что обусловлено квадратичной зависимостью скорости диффузии от градиента потока ($\Delta u \propto 1/\lambda^2$).

2. **Поведение на разрывных данных:**
   - При умеренном шаге $r = 0.4$ все схемы (FTCS, неявный Эйлер, Кранк — Николсон) ведут себя строго монотонно, сглаживая начальный разрыв без паразитных колебаний.
   - При переходе к крупным шагам ($r = 2.0$ и $r = 10.0$) схема Кранка — Николсона генерирует интенсивные пространственно-временные осцилляции шестерёнчатого типа около точек разрыва. Причина заключается в отрицательном факторе усиления старшей гармоники ($G_{\text{CN}} < 0$), приводящем к чередованию её знака на каждом шаге. Абсолютная температура остаётся неотрицательной ($u \ge 0$, глобальный минимум на стержне строго равен 0), однако локальная монотонность грубо нарушается: возникает инверсия градиента (на шаге $n=1$ узел $x=0.38$ оказывается горячее узла $x=0.40$, на шаге $n=2$ — наоборот).
   - Неявная схема Эйлера строго сохраняет принцип максимума ($0 \le u \le 1$) и монотонность при любых $r$ благодаря $L$-устойчивости ($G_{\text{IE}} > 0$), однако при $r = 10$ демонстрирует сильную численную гипердиффузию (завышенное размытие фронта).

3. **О безусловной устойчивости:**
   - Безусловная устойчивость гарантирует лишь ограниченность оператора шага во времени, но не гарантирует монотонности и точности.
   - Схема Кранка — Николсона не обладает свойством $L$-устойчивости ($G_{\text{CN}} \to -1$ при $r \to \infty$), поэтому её применение с крупными шагами на разрывных данных без специального демпфирования приводит к нефизическим колебаниям профиля.